In [2]:
!pip install pyspark==3.3.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.3/281.3 MB 6.1 MB/s eta 0:00:0000:0100:02
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 kB 12.0 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.3.0-py2.py3-none-any.whl size=281764010 sha256=f67b9866c88d581328cb90f7f88c43d7ec42e59827d55778bb0eacf2bd5d0eb1
  Stored in directory: /tmp/pip-ephem-wheel-cache-gxn9osa6/wheels/10/10/4f/65565b95730b1dcb1386e9adc2b26d598cc1ba767ed2d26bca
Successfully built pyspark


In [3]:
import time

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

In [21]:
# Schema for the Kafka JSON message
schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("user_id", IntegerType(), True),
            StructField("username", StringType(), True),
            StructField("password", StringType(), True),
            StructField("email", StringType(), True),
            StructField("created_on", LongType(), True),
            StructField("last_login", LongType(), True)
        ]), True),
        StructField("after", StructType([
            StructField("user_id", IntegerType(), True),
            StructField("username", StringType(), True),
            StructField("password", StringType(), True),
            StructField("email", StringType(), True),
            StructField("created_on", LongType(), True),
            StructField("last_login", LongType(), True)
        ]), True),
        StructField("source", StructType([
            StructField("version", StringType(), True),
            StructField("connector", StringType(), True),
            StructField("name", StringType(), True),
            StructField("ts_ms", LongType(), True),
            StructField("snapshot", StringType(), True),
            StructField("db", StringType(), True),
            StructField("sequence", StringType(), True),
            StructField("ts_us", LongType(), True),
            StructField("ts_ns", LongType(), True),
            StructField("schema", StringType(), True),
            StructField("table", StringType(), True),
            StructField("txId", LongType(), True),
            StructField("lsn", LongType(), True),
            StructField("xmin", LongType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])


In [10]:
spark = SparkSession \
    .builder \
    .appName("Streaming from Kafka") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

In [22]:
streaming_df = spark.readStream\
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "debezium.public.accounts") \
    .option("startingOffsets", "earliest") \
    .load()

json_df = streaming_df.selectExpr("cast(value as string) as value")

# Parse the JSON data
parsedData = json_df.select(from_json(col("value"), schema).alias("data"))

# Flatten the data and select fields
flattenedData = parsedData.select(
    col("data.payload.before.*"),
    col("data.payload.after.*"),
    col("data.payload.ts_ms"),
    col("data.payload.op")
)


In [ ]:
flattenedData.writeStream\
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start() \
    .awaitTermination()